In [22]:
#imports
import torch
import torch.nn as nn
from torch.nn import functional as F
import pickle
import math

In [23]:
#configs
batch_size = 64
block_size = 512 #It determines the maximum number of tokens the model can look at as its context. eg= The wonderful wizard of Oz ...<--------- 64 tokens -------->
max_iters = 1500
learning_rate = 3e-4
eval_iters = 200 #This controls how many batches are used when estimating the training/validation loss.
n_embd = 256
n_head = 16 #The number of attention heads used inside a multi-head attention layer
n_layer =8
dropout = 0.1

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [24]:
#vocab
with open('Data/wizard_of_oz.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()

chars = sorted(set(text))
vocab_size = len(chars)
vocab_size, text[:100]

(81,
 '\ufeff  DOROTHY AND THE WIZARD IN OZ\n\n  BY\n\n  L. FRANK BAUM\n\n  AUTHOR OF THE WIZARD OF OZ, THE LAND OF OZ')

In [25]:
#char_lvl_tokenization
str_to_int = {element : idx for idx, element in enumerate(chars)}
int_to_str = {idx : element for idx, element in enumerate(chars)}

encode = lambda s: [str_to_int[c] for c in s]
decode = lambda i: ''.join([int_to_str[j] for j in i])
encode('a'), decode(encode('a'))

([54], 'a')

In [26]:
#loading_dataset
with open("Data/wizard_of_oz.txt", 'r', encoding= 'utf-8') as f:
    dataset = f.read()

data = torch.tensor(encode(dataset), dtype= torch.long)


In [27]:
#tain_val_split
n = int(0.8 * len(data))
train_data = data[:n]
validation_data = data[n:]

print(f"Train: {len(train_data):,} tokens | Val: {len(validation_data):,} tokens")

Train: 185,847 tokens | Val: 46,462 tokens


In [28]:
#batching
def get_batch(split):
    data = train_data if split == 'train' else validation_data
    idx = torch.randint(len(data) - block_size, (batch_size,))

    #print(idx)

    x = torch.stack([data[i: i + block_size] for i in idx]).to(device)
    y = torch.stack([data[i+1: i + block_size + 1] for i in idx]).to(device)

    return x, y

x, y = get_batch('train')
print('inputs: ' )
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[61, 62, 67,  ..., 57,  1, 59],
        [68,  9,  1,  ...,  9,  3,  1],
        [58,  1, 37,  ..., 27, 68, 76],
        ...,
        [73, 61, 58,  ..., 73, 61, 58],
        [58, 60,  5,  ..., 68,  1, 61],
        [75, 58, 57,  ..., 58,  1, 70]], device='cuda:0')
targets: 
tensor([[62, 67, 60,  ...,  1, 59, 68],
        [ 9,  1, 76,  ...,  3,  1, 71],
        [ 1, 37, 74,  ..., 68, 76, 54],
        ...,
        [61, 58,  1,  ..., 61, 58,  1],
        [60,  5, 73,  ...,  1, 61, 58],
        [58, 57,  1,  ...,  1, 70, 74]], device='cuda:0')


In [29]:
#model
class CausalSelfAttention(nn.Module):
    
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.c_proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = nn.Dropout(dropout)
        
        self.register_buffer('mask', torch.tril(torch.ones(block_size, block_size))
                             .view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        
        qkv = self.c_attn(x)
        q, k, v = qkv.split(C, dim=2)
        
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=dropout if self.training else 0.0)
        
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.c_proj(out)
        out = self.dropout(out)
        return out


class FeedForward(nn.Module):
    
    def __init__(self, n_embd):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):

        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        
        self.token_embedding_table.weight = self.lm_head.weight
        
        self.apply(self._init_weights)
        
        for name, p in self.named_parameters():
            if name.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, index, targets=None):
        B, T = index.shape
        
        tok_emb = self.token_embedding_table(index)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss
    
    def generate(self, index, max_new_tokens, temperature=0.8, top_k=40):

        for _ in range(max_new_tokens):
            index_cond = index[:, -block_size:]
            logits, _ = self.forward(index_cond)
            logits = logits[:, -1, :] / temperature
            
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float('-inf')
            
            probs = F.softmax(logits, dim=-1)
            index_next = torch.multinomial(probs, num_samples=1)
            index = torch.cat((index, index_next), dim=1)
        return index

model = GPTLanguageModel(vocab_size)
m = model.to(device)
print(f"Model parameters: {sum(p.numel() for p in m.parameters()):,}")

Model parameters: 6,462,208


In [30]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [31]:
# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95), weight_decay=0.1)

def get_lr(it):
    warmup_iters = 50
    min_lr = learning_rate / 10
    if it < warmup_iters:
        return learning_rate * (it + 1) / warmup_iters
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return min_lr + coeff * (learning_rate - min_lr)

for iter in range(max_iters):
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr
    
    if iter % 100 == 0:
        losses = estimate_loss()
        print(f"step {iter:4d} | train: {losses['train']:.4f} | val: {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

print(f"\nFinal loss: {loss.item():.4f}")
with open('model-01.pkl', 'wb') as f:
    pickle.dump(model, f)
print('Model saved!')

step    0 | train: 4.4498 | val: 4.4558
step  100 | train: 2.4445 | val: 2.5044
step  200 | train: 2.3690 | val: 2.4367
step  300 | train: 2.2942 | val: 2.3745
step  400 | train: 2.1439 | val: 2.2436
step  500 | train: 1.9910 | val: 2.1093
step  600 | train: 1.8312 | val: 1.9745
step  700 | train: 1.6902 | val: 1.8575
step  800 | train: 1.5796 | val: 1.7729
step  900 | train: 1.4828 | val: 1.7071
step 1000 | train: 1.4201 | val: 1.6690
step 1100 | train: 1.3605 | val: 1.6297
step 1200 | train: 1.3170 | val: 1.6073
step 1300 | train: 1.2851 | val: 1.5879
step 1400 | train: 1.2603 | val: 1.5812

Final loss: 1.3538
Model saved!


In [32]:
prompt = 'Hello Dorothy!'
prompt_tokens = encode(prompt)

context = torch.tensor(prompt_tokens, dtype=torch.long, device=device)
generated_sequence = m.generate(context.unsqueeze(0), max_new_tokens=150, temperature=0.6, top_k=40)[0].tolist()

# Slice the list to start AFTER the prompt tokens
new_tokens_only = generated_sequence[len(prompt_tokens):]

# Decode only the newly generated characters
generated_chars = decode(new_tokens_only)
print(generated_chars)


" where and the little of the rof.

"If country," said one of the boy. "I've bearly the will. "I me no find
by the buggy to the stay," repled the Wiza
